<a href="https://colab.research.google.com/github/positivefunctionIN/Medical_Imaging_using_CNN/blob/main/Model_2_Hybrid_Cnn_CBam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle -q

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
from google.colab import files
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    GlobalAveragePooling2D, GlobalMaxPooling2D,
    Add, Multiply, Concatenate, Activation, Lambda
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)

import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5b435f7615b49cc51191f5ab984c36d2"

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d chest_xray_data

data_path = "/content/chest_xray_data/chest_xray"

IMG_SIZE = 224
BATCH_SIZE = 32
CLASS_NAMES = ['Normal', 'Pneumonia']
NUM_CLASSES = 1


TensorFlow version: 2.20.0
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:14<00:00, 167MB/s]



In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    f"{data_path}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    f"{data_path}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    f"{data_path}/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\n Train samples: {train_generator.samples}")
print(f" Val samples: {val_generator.samples}")
print(f" Test samples: {test_generator.samples}")
print(f" Class mapping: {train_generator.class_indices}")


Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.

 Train samples: 4173
 Val samples: 1043
 Test samples: 624
 Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}


In [3]:
print("🔵 MODEL 1: Custom CNN (From Scratch)")

def build_custom_cnn():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Conv2D(256, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='sigmoid')
    ])
    return model

custom_cnn = build_custom_cnn()
custom_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(" Custom CNN built")
custom_cnn.summary()



🔵 MODEL 1: Custom CNN (From Scratch)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 Custom CNN built


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,263,809 (73.49 MB)

 Trainable params: 19,263,809 (73.49 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
custom_history = custom_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

custom_test_loss, custom_test_acc = custom_cnn.evaluate(test_generator, verbose=0)
print(f"\n Custom CNN Test Accuracy: {custom_test_acc:.2%}")

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 124s 841ms/step - accuracy: 0.8205 - loss: 0.4113 - val_accuracy: 0.8715 - val_loss: 0.2604
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 787ms/step - accuracy: 0.8986 - loss: 0.2490 - val_accuracy: 0.9118 - val_loss: 0.1996
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 774ms/step - accuracy: 0.9008 - loss: 0.2336 - val_accuracy: 0.9147 - val_loss: 0.2064
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 770ms/step - accuracy: 0.9233 - loss: 0.1890 - val_accuracy: 0.9271 - val_loss: 0.1763
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 782ms/step - accuracy: 0.9329 - loss: 0.1701 - val_accuracy: 0.9185 - val_loss: 0.1979
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 774ms/step - accuracy: 0.9387 - loss: 0.1615 - val_accuracy: 0.9367 - val_loss: 0.1480
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 787ms/step - accuracy: 0.9379 - loss: 0.1556 - val_accuracy: 0.9406 - val_loss: 0.1478
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 773ms/step - accuracy: 0.9365 -

In [5]:
from keras import ops
from tensorflow.keras.layers import (
    Layer, Conv2D, Dense, GlobalAveragePooling2D,
    GlobalMaxPooling2D, Multiply, Add,
    Activation, Reshape, Concatenate,
    MaxPooling2D, Dropout, Input
)
from tensorflow.keras.models import Model

In [6]:
class CBAM(Layer):

    def __init__(self, filters, ratio=8):
        super().__init__()

        self.avg_pool = GlobalAveragePooling2D()
        self.max_pool = GlobalMaxPooling2D()

        self.fc1 = Dense(filters//ratio, activation='relu')
        self.fc2 = Dense(filters)

        self.conv = Conv2D(
            1,
            kernel_size=7,
            padding='same',
            activation='sigmoid'
        )

    def call(self, inputs):

        channels = inputs.shape[-1]

        avg = self.avg_pool(inputs)
        avg = Reshape((1,1,channels))(avg)

        mx = self.max_pool(inputs)
        mx = Reshape((1,1,channels))(mx)

        avg = self.fc2(self.fc1(avg))
        mx = self.fc2(self.fc1(mx))

        channel_attention = Activation("sigmoid")(
            Add()([avg,mx])
        )

        x = Multiply()([inputs, channel_attention])

        avg = ops.mean(x, axis=-1, keepdims=True)
        mx = ops.max(x, axis=-1, keepdims=True)

        spatial = Concatenate(axis=-1)([avg,mx])

        spatial = self.conv(spatial)

        output = Multiply()([x, spatial])

        return output

In [7]:
print(" MODEL 2 : Hybrid CNN + CBAM")
inputs = Input(shape=(224,224,3))

x = Conv2D(32,3,activation='relu',padding='same')(inputs)
x = MaxPooling2D()(x)
x = CBAM(32)(x)

x = Conv2D(64,3,activation='relu',padding='same')(x)
x = MaxPooling2D()(x)
x = CBAM(64)(x)

x = Conv2D(128,3,activation='relu',padding='same')(x)
x = MaxPooling2D()(x)
x = CBAM(128)(x)

x = Conv2D(256,3,activation='relu',padding='same')(x)
x = MaxPooling2D()(x)

x = GlobalAveragePooling2D()(x)

x = Dense(512,activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(1,activation='sigmoid')(x)

hybrid_cnn = Model(inputs,outputs)

hybrid_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

hybrid_cnn.summary()

 MODEL 2 : Hybrid CNN + CBAM


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cbam (CBAM)                     │ (None, 112, 112, 32)   │           391 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cbam_1 (CBAM)                   │ (None, 56, 56, 64)     │         1,195 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cbam_2 (CBAM)                   │ (None, 28, 28, 128)    │         4,339 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 526,438 (2.01 MB)

 Trainable params: 526,438 (2.01 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
hybrid_history = hybrid_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[
        EarlyStopping(
            patience=3,
            restore_best_weights=True
        )
    ],
    verbose=1
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 136s 914ms/step - accuracy: 0.7407 - loss: 0.5715 - val_accuracy: 0.7430 - val_loss: 0.5792
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 789ms/step - accuracy: 0.7750 - loss: 0.4188 - val_accuracy: 0.7996 - val_loss: 0.3795
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 104s 792ms/step - accuracy: 0.8193 - loss: 0.3603 - val_accuracy: 0.7833 - val_loss: 0.3773
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 104s 792ms/step - accuracy: 0.8315 - loss: 0.3471 - val_accuracy: 0.8495 - val_loss: 0.3302
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 788ms/step - accuracy: 0.8433 - loss: 0.3303 - val_accuracy: 0.8648 - val_loss: 0.3214
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 146s 818ms/step - accuracy: 0.8610 - loss: 0.3161 - val_accuracy: 0.8667 - val_loss: 0.2996
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 105s 805ms/step - accuracy: 0.8747 - loss: 0.2959 - val_accuracy: 0.8792 - val_loss: 0.2628
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 108s 825ms/step - accuracy: 0.8749 -

In [9]:
hybrid_test_loss, hybrid_test_acc = hybrid_cnn.evaluate(
    test_generator,
    verbose=0
)

print(f"Hybrid CNN Accuracy : {hybrid_test_acc:.2%}")

Hybrid CNN Accuracy : 76.44%


In [10]:
hybrid_cnn.save("hybrid_cnn_cbam.h5")
files.download("hybrid_cnn_cbam.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>